In [1]:
# # 1. Clone the YOLOv5 repo and install requirements
# !git clone https://github.com/ultralytics/yolov5.git
# %cd yolov5
# !pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 7.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 7.6 MB/s eta 0:00:0000:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 14.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 11.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 12.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 11.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 14.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [8]:
import pandas as pd
import os

def convert_annotations(csv_path: str,
                        images_dir: str,
                        labels_dir: str,
                        class_map: dict = {'Graffiti': 0}):
    """
    Reads train_labels.csv and writes one .txt per image into labels_dir,
    in YOLO format: <class_id> <x_center> <y_center> <width> <height>,
    normalized to [0,1].
    """
    df = pd.read_csv(csv_path)
    os.makedirs(labels_dir, exist_ok=True)

    for img_name, group in df.groupby('filename'):
        w = group['width'].iloc[0]
        h = group['height'].iloc[0]
        lines = []
        for _, row in group.iterrows():
            cid = class_map[row['class']]
            x_c = ((row['xmin'] + row['xmax']) / 2) / w
            y_c = ((row['ymin'] + row['ymax']) / 2) / h
            bw = (row['xmax'] - row['xmin']) / w
            bh = (row['ymax'] - row['ymin']) / h
            lines.append(f"{cid} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}")

        txt_path = os.path.join(labels_dir, os.path.splitext(img_name)[0] + '.txt')
        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))

convert_annotations(
    csv_path='/home/admin/Documents/AI_Eng/Week_06/train_labels.csv',
    images_dir='/home/admin/Documents/AI_Eng/Week_06/images/train',
    labels_dir='/home/admin/Documents/AI_Eng/Week_06/labels/train'
)

convert_annotations(
    csv_path='/home/admin/Documents/AI_Eng/Week_06/test_labels.csv',
    images_dir='/home/admin/Documents/AI_Eng/Week_06/images/test',
    labels_dir='/home/admin/Documents/AI_Eng/Week_06/labels/test'
)


In [10]:
import random
import shutil
import os
from pathlib import Path
import yaml

# Paths — adjust as needed
SRC_IMG_TRAIN = Path('/home/admin/Documents/AI_Eng/Week_06/images/train')
SRC_IMG_TEST  = Path('/home/admin/Documents/AI_Eng/Week_06/images/test')
SRC_LBL_TRAIN = Path('/home/admin/Documents/AI_Eng/Week_06/labels/train')
DEST         = Path('/home/admin/Documents/AI_Eng/Week_06/dataset')
DEST_IMG_TR  = DEST/'images'/'train'
DEST_IMG_VA  = DEST/'images'/'val'
DEST_LBL_TR  = DEST/'labels'/'train'
DEST_LBL_VA  = DEST/'labels'/'val'

# Make dirs
for p in [DEST_IMG_TR, DEST_IMG_VA, DEST_LBL_TR, DEST_LBL_VA]:
    p.mkdir(parents=True, exist_ok=True)

# Sample
train_imgs = random.sample(list(SRC_IMG_TRAIN.glob('*.jpg')), 400)
test_imgs  = random.sample(list(SRC_IMG_TEST.glob('*.jpg')),   40)

# Copy images and labels
for img_list, dst_img, dst_lbl in [(train_imgs, DEST_IMG_TR, DEST_LBL_TR),
                                    (test_imgs, DEST_IMG_VA, DEST_LBL_VA)]:
    for img_path in img_list:
        shutil.copy(img_path, dst_img/img_path.name)
        lbl_file = SRC_LBL_TRAIN if img_path.parent.name == 'train' else Path('../labels/test')
        txt_path = lbl_file/(img_path.stem + '.txt')
        if txt_path.exists():
            shutil.copy(txt_path, dst_lbl/(img_path.stem + '.txt'))

# Write dataset YAML
dataset_yaml = {
    'path': str(DEST),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': ['Graffiti']
}
with open(DEST/'graffiti.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f)


In [ ]:
!python train.py   --img 640   --batch 16   --epochs 50   --data ../dataset/graffiti.yaml   --weights yolov5s.pt   --name graffiti_exp1   --cache

In [14]:
import torch
import pandas as pd
import cv2
from pathlib import Path


# Utility: compute IoU between two boxes [x1,y1,x2,y2]
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    inter = interW * interH
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

# Load the trained model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = torch.hub.load(
    'ultralytics/yolov5',
    'custom',
    path='runs/train/graffiti_exp1/weights/best.pt',
    force_reload=True
).to(device)

def load_gt(txt_path: Path, img_w: int, img_h: int):
    boxes = []
    if not txt_path.exists():
        return boxes
    for line in txt_path.read_text().splitlines():
        _, x_c, y_c, w, h = map(float, line.split())
        x1 = (x_c - w/2) * img_w
        y1 = (y_c - h/2) * img_h
        x2 = (x_c + w/2) * img_w
        y2 = (y_c + h/2) * img_h
        boxes.append([x1, y1, x2, y2])
    return boxes

rows = []
for img_path in test_imgs:
    # 1) load raw image and get size
    img0 = cv2.imread(str(img_path))
    h, w = img0.shape[:2]

    # 2) inference
    results = model(img0)                # returns a Detections object
    preds   = results.xyxy[0].cpu().numpy()  # shape: (n,6)

    # 3) load ground-truth
    gt_boxes = load_gt(DEST_LBL_VA/f"{img_path.stem}.txt", w, h)

    # 4) handle no-detections case
    if preds.shape[0] == 0:
        rows.append([img_path.name, 0.0, 0.0])
    else:
        # pick the highest-confidence prediction
        best_idx = preds[:,4].argmax()
        x1,y1,x2,y2,conf,_ = preds[best_idx]
        best_iou = max((iou([x1,y1,x2,y2], gt) for gt in gt_boxes), default=0.0)
        rows.append([img_path.name, float(conf), best_iou])

# 5) save to CSV
df = pd.DataFrame(rows, columns=['image_name','confidence','IoU'])
df.to_csv('iteration1_results.csv', index=False)
print(df.head())


Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /home/admin/.cache/torch/hub/master.zip


YOLOv5 🚀 2025-4-29 Python-3.10.16 torch-2.7.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7836MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
/home/admin/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/home/admin/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/home/admin/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/home/admin/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning

                image_name  confidence       IoU
0  IMG_20180721_091635.jpg    0.928993  0.887810
1  IMG_20180718_105313.jpg    0.760309  0.747227
2  IMG_20180714_102729.jpg    0.928233  0.827125
3  IMG_20180811_083617.jpg    0.683189  0.574603
4  IMG_20180725_114304.jpg    0.899940  0.858678


/home/admin/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/home/admin/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/home/admin/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/home/admin/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/home/admin/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:906:

In [15]:
import numpy as np

for it in range(2, 10):  # up to 10 iterations
    df = pd.read_csv(f'iteration{it-1}_results.csv')
    success = (df['IoU'] >= 0.9).mean()
    print(f"Iteration {it-1}: {success*100:.1f}% ≥ 0.9 IoU")
    if success >= 0.8:
        print("Target reached.")
        break

    # 1) sample new 400 train & 40 val (reuse Segment 3 code, adjust random seed)
    # 2) train starting from prev best.pt:
    !python train.py \
      --img 640 \
      --batch 16 \
      --epochs 30 \
      --data ../dataset/graffiti.yaml \
      --weights runs/train/graffiti_exp{it-1}/weights/best.pt \
      --name graffiti_exp{it} \
      --cache

    # 3) run Segment 5 inference, but save to iteration{it}_results.csv


Iteration 1: 20.0% ≥ 0.9 IoU
2025-04-29 17:18:17.389112: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745911097.399557  191998 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745911097.402484  191998 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745911097.411378  191998 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745911097.411397  191998 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745911097.411399  191998 computation_placer.c

FileNotFoundError: [Errno 2] No such file or directory: 'iteration2_results.csv'